In [56]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [57]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [58]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
# import sys


In [59]:
!pip install langchain-openai

Defaulting to user installation because normal site-packages is not writeable


In [60]:
import sys
!"{sys.executable}" -m pip install -U langchain langchain-community langchain-openai langchain-chroma

Defaulting to user installation because normal site-packages is not writeable


In [61]:
!pip install langchain langchain-community langchain-openai chromadb pypdf

Defaulting to user installation because normal site-packages is not writeable


In [62]:
!pip install langchain-text-splitters

Defaulting to user installation because normal site-packages is not writeable


In [63]:
loader = PyPDFLoader("Thailand Master Sheet.pdf")
docs = loader.load()
len(docs)

11

In [64]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

36

In [65]:
import sys
!"{sys.executable}" -m pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable


In [66]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Free, runs locally, no API key needed
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4321.56it/s]


In [67]:
vector_store = Chroma.from_documents(
    documents=splitted_data, 
    embedding=embeddings 
)

In [68]:
query = "What is this document about?"
data = vector_store.similarity_search(query=query)

In [69]:
len(data)

4

In [70]:
data[0]

Document(id='cd310043-af1b-47e3-825b-5a543cdd98d3', metadata={'page_label': '9', 'page': 8, 'creator': 'Google Sheets', 'source': 'Thailand Master Sheet.pdf', 'creationdate': '', 'total_pages': 11, 'producer': 'PyPDF', 'title': 'Thailand Master Sheet'}, page_content='0118-02009MOHD TABISH BAIG Thai AirAsia Lucknow No\n0221-02443SATYAM TRIPATHI Thai AirAsia Lucknow No\n0921-02640Abhishek kumar Thai AirAsia Lucknow No\n0221-02435Anupam Kushwaha Thai AirAsia Lucknow No\n0721-02564Sunny Sourav Thai AirAsia Lucknow No')

In [71]:
context = ""
for doc in data:
    context += doc.page_content + "\n"
print(context)


0118-02009MOHD TABISH BAIG Thai AirAsia Lucknow No
0221-02443SATYAM TRIPATHI Thai AirAsia Lucknow No
0921-02640Abhishek kumar Thai AirAsia Lucknow No
0221-02435Anupam Kushwaha Thai AirAsia Lucknow No
0721-02564Sunny Sourav Thai AirAsia Lucknow No
0118-02009MOHD TABISH BAIG Thai AirAsia Lucknow No
0221-02443SATYAM TRIPATHI Thai AirAsia Lucknow No
0921-02640Abhishek kumar Thai AirAsia Lucknow No
0221-02435Anupam Kushwaha Thai AirAsia Lucknow No
0721-02564Sunny Sourav Thai AirAsia Lucknow No
0118-02009MOHD TABISH BAIG Thai AirAsia Lucknow No
0221-02443SATYAM TRIPATHI Thai AirAsia Lucknow No
0921-02640Abhishek kumar Thai AirAsia Lucknow No
0221-02435Anupam Kushwaha Thai AirAsia Lucknow No
0721-02564Sunny Sourav Thai AirAsia Lucknow No
0223-02965Shahid Ali Flight 3 - Thai Airways 3.30 AMDelhi No
0123-02954POOJA SHARMA Flight 3 - Thai Airways 3.30 AMDelhi No
1023-03017Saif Ahmad Khan Flight 3 - Thai Airways 3.30 AMDelhi No
0623-02981Dilpreet Singh Flight 3 - Thai Airways 3.30 AMDelhi No



In [ ]:
import sys
!"{sys.executable}" -m pip install langchain-huggingface huggingface-hub

from huggingface_hub import InferenceClient
from langchain_core.language_models.llms import LLM  # ← only this line changed
from typing import Optional, List

client = InferenceClient(token=os.environ["HUGGINGFACEHUB_API_TOKEN"])

class HuggingFaceChatLLM(LLM):
    model: str = "meta-llama/Llama-3.2-1B-Instruct"

    def _call(self, prompt: str, stop: Optional[List[str]] = None) -> str:
        response = client.chat_completion(
            messages=[{"role": "user", "content": prompt}],
            model=self.model,
            max_tokens=512,
        )
        return response.choices[0].message.content

    @property
    def _llm_type(self) -> str:
        return "huggingface_chat"

llm = HuggingFaceChatLLM(model="meta-llama/Llama-3.2-1B-Instruct")

# Test
# print(llm.invoke("Say hello in one word."))

Defaulting to user installation because normal site-packages is not writeable
Hello.


In [77]:
res = llm.invoke(f"""Can you provide me the answer based on
    provided context for my question. context: {context} and question: {query}""")

print(res)

This document appears to be a list of contact information for flights from Thai AirAsia, along with their respective airlines:

- Flight number, airline, and flight time
- Departure city and airport
- Arrival city and airport
- Flight price ( AMDelhi refers to Delhi, India)

The contact information is as follows:

- Flight 3: 
  - From: Shahid Ali, Thai Airways
  - To: Delhi
  - Departure city: No specified, but likely Bangkok or another city in Thailand
  - Arrival city: Delhi
  - Departure time: 3.30 AMDelhi

It seems to be a list of domestic flights in India, with the departure and arrival cities and times.
